# Probe a trained LeWM on OGBCubeDR (RunPod)

Fits **classical linear and MLP probes** on `emb` — the frozen, single-frame
representation a trained LeWM checkpoint produces — and asks, target by
target, what survived encoding. `emb` is not one representation among
several: `LeWM.encode` reads the ViT's CLS token and passes it through
`model.projector` to produce `emb`, and `model.predict` conditions on
exactly that vector. Probing it is probing the one thing the rest of the
model actually uses. `Run.md` §4 describes the digit decal as existing for
exactly this; this notebook covers that and the rest of the task-relevant
state.

Flow: config → apply → GPU check → install → download dataset + verify
checkpoint → pick epoch → extract frozen features → fit probes → read the
results → optional epoch sweep.

**No renderer needed.** Unlike `plan_lewm_ogbcubedr.ipynb` this notebook never
steps MuJoCo, so the `apt-get` GL block and `MUJOCO_GL` do not apply.

### What makes it a clean experiment

- **The encoder is frozen and evaluated once.** Every probe, every target and
  every capacity rung reads the same cached feature matrix, so differences
  between them cannot come from the encoder.
- **One feature, the one that matters.** The ViT encodes every frame
  independently — no temporal mixing happens before the predictor — so a
  frame's `emb` needs no context window to compute, and there is nothing
  upstream of it worth probing instead (a pre-projector token, a pixel
  baseline) since the model itself never uses those.
- **Splits are by episode, not by frame.** Frames 5 env-steps apart in a
  400-step episode are near-duplicates; a frame-level split would leak the
  test set into the train set and inflate every number. (Training itself uses
  a clip-level `random_split` — fine for fitting a world model, wrong for
  measuring one.)
- **Three rungs per target**: constant-predictor baseline → linear probe →
  MLP probe. A score is only ever read as a *difference* against the rung
  below it.
- **Only task-relevant targets are probed** — arm pose, cube layout, the
  digit decal's class — plus the domain-randomization nuisance axes, which
  are visible but irrelevant, and worth checking either way. See
  `scripts/probe/targets.py`.


## 1. Config

Edit the values below. Defaults assume the same `/workspace` network volume
`train_lewm_ogbcubedr.ipynb` used, so the dataset and checkpoint are already
in place and nothing needs downloading.


In [ ]:
import os

# --- repo ---
REPO_ROOT = '/workspace/stable-worldmodel'          # ← edit if you cloned it elsewhere

# --- storage (network volume) ---
STABLEWM_HOME = '/workspace'                        # datasets/, checkpoints/ live directly here

# --- checkpoint to probe ---
OUTPUT_MODEL_NAME = 'lewm_q4_dr'                    # matches the training run name
POLICY_EPOCH = None                                 # ← int to pin an epoch; None = latest on this volume

# --- fallback: only used if the dataset is NOT already on this volume ---
HF_TOKEN = os.environ.get('HF_TOKEN', '')
HF_DATASET_REPO_ID = '<your-hf-username-or-org>/ogbench-cube-quadruple-domain-randomized-expert'  # ← edit me

# --- experiment scale ---
# Episodes are the unit of the split; frames are sampled inside them.
#
# **Do not under-sample episodes.** Every episode re-draws lighting, camera
# angle, cube colours, floor/wall materials and the backdrop, so episode-level
# appearance is the dominant direction of variation in the features. With ~100
# train episodes a linear probe can fit episode identity and every
# within-episode target reads ~0 on held-out episodes -- measured, not
# hypothetical. 1000 episodes is the smallest setting that gave a usable
# signal; raise it before raising FRAMES_PER_EPISODE, which adds correlated
# samples rather than independent ones.
TRAIN_EPISODES = 1000
VAL_EPISODES = 150
TEST_EPISODES = 250
FRAMES_PER_EPISODE = 20
SEED = 0
IMG_SIZE = 224

# --- compute ---
BATCH_SIZE = 128
NUM_WORKERS = 6
DTYPE = 'float32'       # 'bfloat16' matches training precision and is ~2x faster

# --- derived, don't edit ---
DATASET_NAME = 'ogbench/cube_quadruple_dr_expert.lance'
DATASET_DIR = os.path.join(STABLEWM_HOME, 'datasets', 'ogbench', 'cube_quadruple_dr_expert.lance')
CHECKPOINT_DIR = os.path.join(STABLEWM_HOME, 'checkpoints', OUTPUT_MODEL_NAME)
RESULTS_ROOT = os.path.join(STABLEWM_HOME, 'probing', OUTPUT_MODEL_NAME)


## 2. Apply config


In [ ]:
os.environ['STABLEWM_HOME'] = STABLEWM_HOME
os.environ['HF_TOKEN'] = HF_TOKEN

os.makedirs(STABLEWM_HOME, exist_ok=True)
os.makedirs(RESULTS_ROOT, exist_ok=True)

os.chdir(REPO_ROOT)  # os.chdir (not `%cd`) so it's identical whether run fresh or after a kernel restart

print('cwd            =', os.getcwd())
print('STABLEWM_HOME  =', os.environ['STABLEWM_HOME'])
print('DATASET_DIR    =', DATASET_DIR)
print('CHECKPOINT_DIR =', CHECKPOINT_DIR)
print('RESULTS_ROOT   =', RESULTS_ROOT)
!df -h /workspace


## 3. GPU check

Not strictly required — everything runs on CPU — but feature extraction is a
ViT-small forward over `(TRAIN+VAL+TEST episodes) x FRAMES_PER_EPISODE`
frames, which is minutes on a GPU and hours on a CPU. Probe fitting itself is
seconds either way.


In [ ]:
import torch

print('torch', torch.__version__)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU visible — extraction will run on CPU. Lower TRAIN_EPISODES / '
          'FRAMES_PER_EPISODE, or set DTYPE = "bfloat16", if this is a test run.')


## 4. Install dependencies

Same scoped install as training (`Run.md` §3) — the probing code needs the
encoder and the Lance reader, nothing that renders.


In [ ]:
%pip install -q -e '.[train,format]' huggingface_hub


## 5. Verify (or fetch) the dataset, and verify the checkpoint


In [ ]:
if os.path.isdir(DATASET_DIR) and os.listdir(DATASET_DIR):
    print('Dataset already present:', DATASET_DIR)
else:
    assert HF_DATASET_REPO_ID and '<' not in HF_DATASET_REPO_ID, (
        'Dataset missing and HF_DATASET_REPO_ID is not set — edit the config cell.'
    )
    os.makedirs(DATASET_DIR, exist_ok=True)
    !hf download "$HF_DATASET_REPO_ID" --repo-type dataset --local-dir "$DATASET_DIR"


In [ ]:
import glob

assert os.path.isdir(CHECKPOINT_DIR), (
    f'{CHECKPOINT_DIR} does not exist. Copy the trained run onto this volume '
    '(e.g. from baseline_lewm_08_2026/checkpoints/) before probing.'
)
assert os.path.exists(os.path.join(CHECKPOINT_DIR, 'config.json')), (
    f'{CHECKPOINT_DIR} is missing config.json — the architecture is read from it.'
)
print(sorted(os.path.basename(p) for p in glob.glob(os.path.join(CHECKPOINT_DIR, '*.pt'))))


## 6. Pick the checkpoint epoch


In [ ]:
import re

ckpt_files = glob.glob(os.path.join(CHECKPOINT_DIR, 'weights_epoch_*.pt'))
assert ckpt_files, f'No weights_epoch_*.pt found in {CHECKPOINT_DIR}'

epochs = sorted(int(re.search(r'weights_epoch_(\d+)\.pt$', f).group(1)) for f in ckpt_files)
epoch = POLICY_EPOCH if POLICY_EPOCH is not None else epochs[-1]
assert epoch in epochs, f'weights_epoch_{epoch}.pt not found; available: {epochs}'

CHECKPOINT = f'{OUTPUT_MODEL_NAME}/weights_epoch_{epoch}.pt'
print('Available epochs:', epochs)
print('Probing          :', CHECKPOINT)


## 7. Load the probing code

`scripts/probe/` holds the experiment: `targets.py` (the label registry),
`features.py` (frozen-feature extraction), `fit.py` (the read-outs) and
`run_probing.py` (the same thing as a CLI). Read each module's docstring —
they carry the design decisions this notebook only summarizes.


In [ ]:
import sys

sys.path.insert(0, os.path.join(REPO_ROOT, 'scripts', 'probe'))

import features as ft
import fit as fitting
import targets as tg

print(f'{len(tg.TARGETS)} targets registered:')
for group in tg.GROUPS:
    names = [t.name for t in tg.TARGETS if t.group == group]
    print(f'  {group:<9s} ({len(names)}) {", ".join(names)}')
print()
print('feature source: emb (projector output of the ViT CLS token — what the predictor consumes)')


Which targets to run. Everything is the default; trim to `groups=['state']`
for a fast first look.

**One statistical caveat, reported per target as `n_train_effective`.** Five
labels are constant within an episode — every domain-randomization axis
(`digit_value`, `floor_material`, `wall_material`, `floor_rgb`,
`light_pos`). Sampling 20 frames from one episode gives 20 *identical*
labels, so their effective training size is the number of episodes, not the
number of frames: 1,000 rather than 20,000. Their scores are the noisiest in
the table, and they are the targets for which an episode-level split is not
merely good practice but the only split that means anything.


In [ ]:
PROBE_TARGETS = tg.select_targets()          # or: tg.select_targets(groups=['state'])
LABEL_COLUMNS = tg.required_columns(PROBE_TARGETS)

print(f'{len(PROBE_TARGETS)} targets')
print(f'{len(LABEL_COLUMNS)} label columns to load: {LABEL_COLUMNS}')


## 8. Extract frozen features

Runs the encoder once and caches everything to `.npz`: `emb` for every
sampled frame, every label column, and the frame's provenance (episode +
frame index). Re-running the cell reuses the cache — delete the file to
force a re-encode.


In [ ]:
from pathlib import Path


def extract_config(checkpoint):
    return ft.ExtractConfig(
        dataset_name=DATASET_NAME,
        checkpoint=checkpoint,
        img_size=IMG_SIZE,
        episodes={'train': TRAIN_EPISODES, 'val': VAL_EPISODES, 'test': TEST_EPISODES},
        frames_per_episode=FRAMES_PER_EPISODE,
        seed=SEED,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        dtype=DTYPE,
    )


def get_features(tag, checkpoint):
    cache = Path(RESULTS_ROOT) / f'features_{tag}.npz'
    if cache.exists():
        print(f'Reusing {cache}')
        return ft.load_features(cache)
    payload = ft.extract(extract_config(checkpoint), LABEL_COLUMNS)
    ft.save_features(cache, payload)
    return payload


TRAINED = f'trained_epoch{epoch}'
payload = get_features(TRAINED, CHECKPOINT)
meta = payload['meta']

print()
print('frames per split :', meta['num_frames'])
print('feature dim      :', meta['feature_dim'])


The episode split is disjoint by construction — worth asserting rather than
trusting, since it is the one mistake that would quietly invalidate every
number below.


In [ ]:
eps = meta['episodes']
train_eps, val_eps, test_eps = (set(eps[s]) for s in ('train', 'val', 'test'))
assert not (train_eps & val_eps), 'train/val episode overlap'
assert not (train_eps & test_eps), 'train/test episode overlap'
assert not (val_eps & test_eps), 'val/test episode overlap'
print(f'episodes: {len(train_eps)} train / {len(val_eps)} val / {len(test_eps)} test, pairwise disjoint')


## 9. Fit the probes

Three rungs per target. Regression linear probes are solved in closed form
with a ridge penalty chosen on validation — no optimizer to blame for a low
score. Classification probes and all MLP probes are fitted with AdamW and
early-stopped on validation.

Cost scales as `len(PROBE_TARGETS) x 3`; at the defaults that is ~33 fits,
seconds on a GPU.


In [ ]:
fit_cfg = fitting.FitConfig(
    probes=('baseline', 'linear', 'mlp'),
    mlp_hidden_dim=512,
    mlp_layers=1,
    epochs=200,
    patience=25,
    seed=SEED,
)

rows, _ = fitting.fit_all(payload, PROBE_TARGETS, fit_cfg, progress=True)
for row in rows:
    row['run'] = TRAINED

print(f'\n{len(rows)} fits done')


In [ ]:
import json

import pandas as pd

df = pd.DataFrame(rows)
df.to_csv(os.path.join(RESULTS_ROOT, f'probe_results_epoch{epoch}.csv'), index=False)
with open(os.path.join(RESULTS_ROOT, f'probe_results_epoch{epoch}.json'), 'w') as f:
    json.dump({'rows': rows, 'manifest': meta}, f, indent=2)

print('saved to', RESULTS_ROOT)
df.head()


## 10. Results

### 10.1 Score per target × probe

`R2` for regression, accuracy for classification. Read each row against its
`baseline` column: `mlp - linear` is information present but not linearly
decodable.


In [ ]:
def headline(frame):
    sub = frame[frame.run == TRAINED]
    wide = sub.pivot_table(index='target', columns='probe', values='score')

    order = [t.name for t in PROBE_TARGETS if t.name in wide.index]
    out = pd.DataFrame(index=order)
    out['group'] = [tg.TARGETS_BY_NAME[t].group for t in order]
    out['metric'] = ['acc' if tg.TARGETS_BY_NAME[t].kind == 'classification' else 'R2'
                     for t in order]
    out['eff_N'] = [
        int(sub[sub.target == t].n_train_effective.iloc[0]) for t in order
    ]
    out['baseline'] = wide['baseline'].reindex(order)
    out['linear'] = wide['linear'].reindex(order)
    out['mlp'] = wide['mlp'].reindex(order)
    out['mlp - linear'] = out['mlp'] - out['linear']
    return out


headline(df).round(3)


### 10.2 Regression error in physical units

R² answers "how much of the variance", MAE answers "how wrong, in metres".
Both matter: a cube-position probe at R² = 0.9 with a 2 cm MAE is above the
planner's 4 cm subgoal tolerance (`Run.md` §7) and one at 0.9 with 5 mm is
not.


In [ ]:
reg = df[(df.run == TRAINED) & (df.kind == 'regression') & (df.probe != 'baseline')]
units = {t.name: t.units for t in PROBE_TARGETS}
err = reg.pivot_table(index=['target', 'probe'], values='mae')
err.insert(0, 'units', [units[t] for t, _ in err.index])
err.round(4)


### 10.3 Within-episode ceiling

The episode split asks a hard question: decode the state of an episode whose
lighting, camera angle, cube colours and materials the probe has never seen.
Re-fitting the *same* features on a deliberately leaky **frame-level** split
— same episodes on both sides — separates two very different failure modes:

- **high leaky, low episode-split** → the information is in the
  representation, but entangled with episode-level appearance. More training
  episodes, or an appearance-invariant encoder, would help.
- **low both** → the information is not in the representation at all.

This is a diagnostic, not a result: the leaky number is inflated by
construction and must never be quoted on its own. It is meaningless for the
episode-constant targets (a leaky split hands the probe the answer), so they
are excluded.


In [ ]:
import numpy as np


def leaky_reference(payload, probe_targets, seed=0):
    """Refit on a frame-level split of the pooled frames (leaky on purpose)."""
    def pool(kind):
        if kind == 'features':
            return np.concatenate([payload['features'][s] for s in ft.SPLITS])
        return {
            k: np.concatenate([payload[kind][s][k] for s in ft.SPLITS])
            for k in payload[kind]['train']
        }

    feats, labs = pool('features'), pool('labels')
    n = len(feats)
    perm = np.random.default_rng(seed).permutation(n)
    idx = {
        'train': perm[: int(0.7 * n)],
        'val': perm[int(0.7 * n) : int(0.8 * n)],
        'test': perm[int(0.8 * n) :],
    }
    cfg = fitting.FitConfig(probes=('baseline', 'linear'), seed=seed)

    out = {}
    for target in probe_targets:
        if target.episode_constant:
            continue
        y_all = tg.build_labels(target, labs, step=0)
        x = {s: feats[i] for s, i in idx.items()}
        y = {s: y_all[i] for s, i in idx.items()}
        _, score, _, _, _ = fitting.fit_one('linear', x, y, target, cfg, 'cpu')
        out[target.name] = score
    return out


leaky = leaky_reference(payload, PROBE_TARGETS)
episode_split_scores = df[
    (df.run == TRAINED) & (df.probe == 'linear')
].set_index('target').score

ceiling = pd.DataFrame({
    'episode split': episode_split_scores.reindex(leaky.keys()),
    'leaky (frame split)': pd.Series(leaky),
})
ceiling['generalization gap'] = (
    ceiling['leaky (frame split)'] - ceiling['episode split']
)
ceiling['group'] = [tg.TARGETS_BY_NAME[t].group for t in ceiling.index]
ceiling.round(3)


### 10.4 Plot


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(12, 6), sharey=True)

for ax, (probe, title) in zip(axes, [('linear', 'Linear probe'), ('mlp', 'MLP probe')]):
    sub = df[(df.run == TRAINED) & (df.probe == probe)]
    scores = sub.set_index('target').score
    order = [t.name for t in PROBE_TARGETS if t.name in scores.index][::-1]

    y = np.arange(len(order))
    colors = {'state': 'tab:blue', 'nuisance': 'tab:orange'}
    bar_colors = [colors[tg.TARGETS_BY_NAME[t].group] for t in order]
    ax.barh(y, scores.reindex(order).values, color=bar_colors)
    ax.set_yticks(y)
    ax.set_yticklabels([
        f'{t} [{tg.TARGETS_BY_NAME[t].group[:4]}]' for t in order
    ])
    ax.axvline(0, color='k', lw=0.8)
    ax.set_xlim(min(-0.15, float(np.nanmin(scores.values)) - 0.05), 1.0)
    ax.set_xlabel('R2 (regression) / accuracy (classification)')
    ax.set_title(title)
    ax.grid(axis='x', alpha=0.3)

fig.suptitle(f'Probing {TRAINED} on emb: what is decodable')
fig.tight_layout()
plt.show()


## 11. Optional: representation quality over training

Same probes, several epochs. Answers whether the linear decodability of the
state tracks the training loss, or saturates early. Extraction is the cost —
one pass per epoch — so this cell reuses caches and is safe to interrupt and
resume.


In [ ]:
RUN_EPOCH_SWEEP = False        # ← set True
SWEEP_EPOCHS = [1, 4, 8, epoch]
SWEEP_TARGETS = tg.select_targets(groups=['state'])

if RUN_EPOCH_SWEEP:
    sweep_rows = []
    sweep_cfg = fitting.FitConfig(probes=('baseline', 'linear'), seed=SEED)
    for ep in SWEEP_EPOCHS:
        assert ep in epochs, f'weights_epoch_{ep}.pt is not on this volume'
        tag = f'trained_epoch{ep}'
        ep_payload = get_features(tag, f'{OUTPUT_MODEL_NAME}/weights_epoch_{ep}.pt')
        run_rows, _ = fitting.fit_all(
            ep_payload, SWEEP_TARGETS, sweep_cfg, progress=False,
        )
        for row in run_rows:
            row['run'], row['epoch'] = tag, ep
        sweep_rows.extend(run_rows)
        print(f'epoch {ep}: done')

    sweep = pd.DataFrame(sweep_rows)
    sweep = sweep[sweep.probe == 'linear']
    curve = sweep.pivot_table(index='epoch', columns='target', values='score')

    fig, ax = plt.subplots(figsize=(9, 5))
    curve.plot(marker='o', ax=ax)
    ax.set_xlabel('checkpoint epoch')
    ax.set_ylabel('linear probe R2 / acc on emb')
    ax.set_title('Linear decodability of the state over training')
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8, ncol=2)
    plt.show()
    display(curve.round(3))


## 12. How to read this

1. `score > baseline` — otherwise the probe learned the marginal, nothing
   else.
2. **`mlp − linear`** is information present but not linearly decodable.
   Large gaps are a real finding, not noise — but check `hp_weight_decay`
   and `val_score` before believing a gap smaller than the spread across
   seeds.
3. **Check `eff_N` before believing a nuisance-target score.** The five
   episode-constant labels have one independent sample per *episode*, so at
   the default scale their effective training size is 1,000, not 20,000 —
   and their scores are correspondingly noisy. A within-episode target at
   the same score is backed by 20× the data.
4. **Everything reading ~0 usually means too few episodes, not a dead
   representation.** Domain randomization makes episode-level appearance the
   dominant direction in the features; with ~100 training episodes a linear
   probe fits episode identity and every within-episode target lands at
   R² ≈ 0 on held-out episodes. Section 10.3's leaky reference distinguishes
   that case from a representation that genuinely lacks the information.
5. **Known label noise.** A cube can sit on top of the digit decal
   (`Run.md` §4) — `privileged/digit_0_value` still reports the digit in
   full, so the `digit_value` ceiling is below 1.0.

### Same thing from a terminal

```bash
python scripts/probe/run_probing.py \
    --checkpoint lewm_q4_dr/weights_epoch_11.pt \
    --out $STABLEWM_HOME/probing/lewm_q4_dr
```
